In [1]:
import pandas as pd

In [2]:
df = pd.read_excel('./part.xlsx',
                   header=0,
                   dtype=str).fillna('')

list_type_code = df['Type_Code'].tolist()
list_part_key = df['Part_Key'].tolist()

print('总数量：' + str(len(list_part_key)))
list_part_key

总数量：1


['8304960']

In [3]:
df = pd.DataFrame()

df

""


In [4]:
from urllib.parse import quote
import requests
import json

In [5]:
i = 0

type_code = list_type_code[i]
part_key = list_part_key[i]

data = 'func=getbuyersguide&payload=' + quote('{"partData":{"listing_data_essential":{"parttype":"') + type_code + quote('","partkey":"') + part_key + quote('"}}}') + '&api_json_request=1&sctchecked=1&scbeenloaded=true&curCartGroupID='

resp = requests.post('https://www.rockauto.com/catalog/catalogapi.php',
                     data=data,
                     headers={'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8'}).text

resp = json.loads(resp)

resp

{'buyersguidepieces': {'body': '<div class="buyersguide-nested"><div style="padding: .5em;"><table class="nobmp" style="width: 100%;"><tr><td class="altrow-a-0">MAZDA</td><td class="altrow-a-0">3</td><td class="altrow-a-0">2004-2007</td></tr><tr><td class="altrow-a-1">MAZDA</td><td class="altrow-a-1">5</td><td class="altrow-a-1">2006-2007</td></tr><tr><td class="altrow-a-0">MAZDA</td><td class="altrow-a-0">6</td><td class="altrow-a-0">2003-2007</td></tr></table><div>Please refer to catalog for application details.</div></div></div>',
  'title': "Buyer's Guide :   Timing Chain & Component Kit"},
 'groupindex': None,
 'collected_javascript': '/**UQ:B3A74BE0**/\nwindow.top.parent.window["_nck"] = "8LVLUxTaILBT+tLxkg8+3Ag7\\/0bbOhvn7P3rkFbL80\\/ismhHyfCiO07pwNUCaIW3afbJMKR0uqi1pNnUlRgBN5CwkZnWFYwm7DtiVw8cl6t1DDIlw0OPK8L1xqUF2GhWjP3ue6n1ZKA+suUZbc5NUvNKDX6iwKsojQo1gmBpIbo=";\n'}

In [6]:
from bs4 import BeautifulSoup

In [7]:
html = resp['buyersguidepieces']['body']

soup = BeautifulSoup(html, 'lxml')

with open('./vehicle.html', 'w', encoding='UTF-8') as file:
    file.write(str(soup))

print(soup.prettify())

<html>
 <body>
  <div class="buyersguide-nested">
   <div style="padding: .5em;">
    <table class="nobmp" style="width: 100%;">
     <tr>
      <td class="altrow-a-0">
       MAZDA
      </td>
      <td class="altrow-a-0">
       3
      </td>
      <td class="altrow-a-0">
       2004-2007
      </td>
     </tr>
     <tr>
      <td class="altrow-a-1">
       MAZDA
      </td>
      <td class="altrow-a-1">
       5
      </td>
      <td class="altrow-a-1">
       2006-2007
      </td>
     </tr>
     <tr>
      <td class="altrow-a-0">
       MAZDA
      </td>
      <td class="altrow-a-0">
       6
      </td>
      <td class="altrow-a-0">
       2003-2007
      </td>
     </tr>
    </table>
    <div>
     Please refer to catalog for application details.
    </div>
   </div>
  </div>
 </body>
</html>



In [8]:
from lxml import etree

In [9]:
html = etree.HTML(str(soup))

list_vehicle = html.xpath('//div[contains(@class, "buyersguide-nested")]/descendant::table[contains(@class, "nobmp")]/tr')

vehicle = '\n'.join([' '.join(_.xpath('./td/text()')).strip() for _ in list_vehicle])

print(vehicle)

MAZDA 3 2004-2007
MAZDA 5 2006-2007
MAZDA 6 2003-2007


In [10]:
df_temp = pd.DataFrame([{'status': 'ok',
                         'Part_Key': part_key,
                         'Vehicle': vehicle}])

df = pd.concat([df, df_temp], ignore_index=True).fillna('')

df

,status,Part_Key,Vehicle
0,ok,8304960,MAZDA 3 2004-2007\nMAZDA 5 2006-2007\nMAZDA 6 ...
